# GitLab Projects Export — v5

## Three-table design

| Table | Content | Key |
|---|---|---|
| `gwma_ui_projects` | One row per project with `package.json` | `project_id` |
| `gwma_ui_dependencies` | One row per dependency per project | `project_id` |
| `gwma_ui_imports` | One row per import statement per file | `project_id` |

Join all three on `project_id`.

## Sampling flag

| `SAMPLE_MODE` | Behaviour |
|---|---|
| `False` (default) | Full run — processes all projects, writes to the three main tables |
| `True` | Asks for a single `project_id`, runs all phases for that project only, writes to `*_sample` tables |

**Sample tables:** `gwma_ui_projects_sample`, `gwma_ui_dependencies_sample`, `gwma_ui_imports_sample`

## Flow (SAMPLE_MODE = False)
- **Phase 1 gate**: Run? Yes → scan GitLab, recreate `gwma_ui_projects`. No → skip.
- **Phase 2 gate**: Run? Yes → parse `package.json` deps + scan source files. No → exit.
- Phase 2 always reads from `gwma_ui_projects` (either freshly written or pre-existing).


## 1 · Imports & logging

In [ ]:
import getpass, json, logging, re, sys, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutureTimeout
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple
import gitlab, psycopg2, psycopg2.extras, requests
from requests.adapters import HTTPAdapter

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S', force=True)
logger = logging.getLogger('gitlab_export')
logging.getLogger('urllib3').setLevel(logging.WARNING)
logging.getLogger('gitlab').setLevel(logging.WARNING)
logger.info('GitLab Projects Export  v5')


## 2 · Configuration

In [ ]:
GITLAB_URL='https://devcloud.ubs.net'
GROUP_PATH='ubs/gwma'
PER_PAGE=100; MAX_WORKERS=10; FILE_WORKERS=5
MAX_CONNECTIONS=10; GITLAB_TIMEOUT=(10,30); FUTURE_TIMEOUT=120; PROGRESS_EVERY=5
UPDATED_SINCE=datetime(2025,1,1,tzinfo=timezone.utc)
# All blob files scanned — no extension filter

DB_SCHEMA='sandbox_prj_smart_insights'
DB_OWNER='erd_gpdb_prj_smart_insights'
DB_GRANT='erd_gpdb_prj_smart_insights_ro'
DB_BATCH=200
# Set True to test a single project_id; writes to *_sample tables.
# Set False for full export against the main tables.
SAMPLE_MODE=False
_TBL_SUFFIX='_sample' if SAMPLE_MODE else ''
TBL_PROJECTS=f'{DB_SCHEMA}.gwma_ui_projects{_TBL_SUFFIX}'
TBL_DEPS    =f'{DB_SCHEMA}.gwma_ui_dependencies{_TBL_SUFFIX}'
TBL_IMPORTS =f'{DB_SCHEMA}.gwma_ui_imports{_TBL_SUFFIX}'

PROJECTS_COLS=[('project_id','BIGINT'),('name','TEXT'),('path','TEXT'),
    ('path_with_namespace','TEXT'),('group_path','TEXT'),('web_url','TEXT'),
    ('description','TEXT'),('visibility','TEXT'),('archived','BOOLEAN'),
    ('created_at','TIMESTAMP WITH TIME ZONE'),('last_activity_at','TIMESTAMP WITH TIME ZONE'),
    ('updated_at','TIMESTAMP WITH TIME ZONE'),('default_branch','TEXT'),
    ('forks_count','INTEGER'),('star_count','INTEGER'),('open_issues_count','INTEGER'),
    ('team_name','TEXT'),('package_json','TEXT')]
DEPS_COLS   =[('project_id','BIGINT'),('name','TEXT'),('path','TEXT'),
    ('tag','TEXT'),('dependency','TEXT'),('version','TEXT')]
IMPORTS_COLS=[('project_id','BIGINT'),('name','TEXT'),('path','TEXT'),
    ('import_filename','TEXT'),('import_file_url','TEXT'),('import_statement','TEXT')]

PROJECTS_COL_NAMES=[c[0] for c in PROJECTS_COLS]
DEPS_COL_NAMES    =[c[0] for c in DEPS_COLS]
IMPORTS_COL_NAMES =[c[0] for c in IMPORTS_COLS]

# Each token uses [^\S\n] (horiz. space only) — never crosses a newline.
ANY_IMPORT_RE=re.compile(
    r'^[^\S\n]*import'
    r'(?:[^\S\n]+(?:'
        r'[\w\$_][\w\$_]*'
        r'(?:[^\S\n]*,[^\S\n]*(?:\*[^\S\n]+as[^\S\n]+[\w\$_]+|\{[^\}\n]*\}))?'
        r'|[^\S\n]*\*[^\S\n]+as[^\S\n]+[\w\$_]+'
        r'|\{[^\}\n]*\}'
    r')[^\S\n]+from)?'
    r"[^\S\n]*['\"]([^'\"]+)['\"]"
    r'[^\S\n]*;?[^\n]*', re.MULTILINE)

_stats_lock=threading.Lock()
run_stats={k:0 for k in ['total_in_group','recent_projects','skipped_too_old',
    'has_package_json','skipped_no_package_json','p1_errors','p1_timeouts',
    'dep_rows_total','source_files_fetched','source_files_empty','source_files_error',
    'import_rows_total','p2_errors','p2_timeouts']}
def _inc(k,n=1):
    with _stats_lock: run_stats[k]+=n
_gitlab_sem=threading.Semaphore(MAX_CONNECTIONS)
logger.info('Config ready')


## 3 · Credentials

In [ ]:
private_token = getpass.getpass('Enter your GitLab private token: ')

class _TimeoutSession(requests.Session):
    """requests.Session subclass that injects a default timeout on every request.
    Using a subclass (not monkey-patching) means the override survives
    across cell re-runs and threaded use."""
    def request(self, method, url, **kwargs):
        kwargs.setdefault('timeout', GITLAB_TIMEOUT)
        return super().request(method, url, **kwargs)

_session = _TimeoutSession()
_adapter = HTTPAdapter(max_retries=1,
                        pool_connections=MAX_CONNECTIONS,
                        pool_maxsize=MAX_CONNECTIONS)
_session.mount('https://', _adapter)
_session.mount('http://',  _adapter)

client = gitlab.Gitlab(GITLAB_URL, private_token=private_token, session=_session)
logger.info('AUTH | GitLab client ready')

db_password = getpass.getpass('Enter Password for DB User: ')
DB_CONFIG = {'host': 'greenplum-rdsp.zur.swissbank.com', 'port': 5432,
             'dbname': 'gprdsp', 'user': 'ds_rdsp_dev', 'password': db_password}
logger.info('DB credentials stored')


## 4 · Utility helpers

In [ ]:
def _ask(q):
    return input(f'\n  {q} (yes/no): ').strip().lower() in ('yes','y')

def _parse_dt(s):
    if not s: return None
    try: return datetime.fromisoformat(s.replace('Z','+00:00'))
    except: return None

def is_recent(p):
    dt=_parse_dt(getattr(p,'updated_at',None) or '')
    return True if dt is None else dt>=UPDATED_SINCE

def extract_team_name(url):
    try:
        segs=url.rstrip('/').split('//',1)[-1].split('/')[1:]
        return segs[-2] if len(segs)>=2 else ''
    except: return ''

def _raw_file(project,path,ref):
    """Fetch file as UTF-8. Normalises \\r\\n -> \\n so the regex never merges lines."""
    with _gitlab_sem:
        try:
            raw=project.files.raw(file_path=path,ref=ref).decode('utf-8',errors='replace')
            return raw.replace('\r\n','\n').replace('\r','\n')
        except: return None

def _repo_tree_pages(project,ref):
    items,page=[],1
    while True:
        with _gitlab_sem:
            try: batch=project.repository_tree(ref=ref,recursive=True,per_page=PER_PAGE,page=page,get_all=False)
            except Exception as e: logger.warning('TREE page=%d err %s %s',page,project.path_with_namespace,e); break
        if not batch: break
        items.extend(batch)
        if len(batch)<PER_PAGE: break
        page+=1
    return items

def _coerce(v):
    if v is None: return None
    if isinstance(v,float) and v!=v: return None
    if isinstance(v,str) and v.strip()=='': return None
    return v

logger.info('Helpers defined')


## 5 · Greenplum helpers

In [ ]:
def _get_conn():
    return psycopg2.connect(host=DB_CONFIG['host'],port=DB_CONFIG['port'],
        dbname=DB_CONFIG['dbname'],user=DB_CONFIG['user'],password=DB_CONFIG['password'])

def db_test_connection():
    try:
        conn=_get_conn()
        with conn.cursor() as cur: cur.execute('SELECT 1')
        conn.close(); logger.info('DB connection test OK'); return True
    except Exception as e: logger.error('DB connection FAILED: %s',e); return False

def _create_table(conn,fqtable,cols,dist='project_id'):
    col_defs=',\n    '.join(f'"{c}" {t}' for c,t in cols)
    with conn.cursor() as cur:
        cur.execute(f'DROP TABLE IF EXISTS {fqtable};')
        sql=f'CREATE TABLE {fqtable} (\n    {col_defs}\n) DISTRIBUTED BY ({dist});'
        logger.info('DB CREATE TABLE %s',fqtable)
        cur.execute(sql)
        cur.execute(f'ALTER TABLE {fqtable} OWNER TO {DB_OWNER};')
        cur.execute(f'GRANT SELECT ON {fqtable} TO {DB_GRANT};')
    conn.commit(); logger.info('DB table ready: %s',fqtable)

def _bulk_insert(conn,fqtable,col_names,rows):
    if not rows: logger.warning('DB no rows for %s',fqtable); return 0
    col_list=', '.join(f'"{c}"' for c in col_names)
    template='('+', '.join(['%s']*len(col_names))+')'
    sql=f'INSERT INTO {fqtable} ({col_list}) VALUES %s'
    tuples=[tuple(_coerce(r.get(c)) for c in col_names) for r in rows]
    logger.info('DB INSERT %d rows into %s',len(tuples),fqtable)
    logger.info('DB sample[0]: %s',dict(zip(col_names,tuples[0])) if tuples else None)
    inserted=0
    with conn.cursor() as cur:
        for s in range(0,len(tuples),DB_BATCH):
            batch=tuples[s:s+DB_BATCH]
            psycopg2.extras.execute_values(cur,sql,batch,template=template)
            inserted+=len(batch)
            logger.info('DB %s : committed %d/%d',fqtable,inserted,len(tuples))
    conn.commit(); logger.info('DB INSERT done: %d rows in %s',inserted,fqtable)
    return inserted

def db_load_projects():
    logger.info('DB loading from %s ...',TBL_PROJECTS)
    conn=_get_conn()
    with conn.cursor(cursor_factory=psycopg2.extras.DictCursor) as cur:
        cur.execute(f'SELECT project_id,name,path,path_with_namespace,web_url,default_branch '
                    f'FROM {TBL_PROJECTS} ORDER BY project_id')
        rows=[dict(r) for r in cur.fetchall()]
    conn.close(); logger.info('DB loaded %d projects',len(rows)); return rows

logger.info('DB helpers defined')


## 6 · Phase 1 worker — fetch project + check package.json

In [ ]:
def fetch_project_with_pkg(proj_ref):
    ns=proj_ref.path_with_namespace
    with _gitlab_sem:
        try: project=client.projects.get(proj_ref.id)
        except Exception as e:
            logger.error('P1 project.get FAILED %s %s',ns,e); _inc('p1_errors'); return None
    ref=project.default_branch or 'main'
    ns_d=project.namespace or {}
    with _gitlab_sem:
        try: project.files.raw(file_path='package.json',ref=ref); has_pkg=True
        except: has_pkg=False
    if not has_pkg:
        _inc('skipped_no_package_json'); return None
    logger.info('P1 YES pkg | %s',project.path_with_namespace)
    _inc('has_package_json')
    return {
        '_project':project,'_ref':ref,
        'project_id':int(project.id),'name':project.name,'path':project.path,
        'path_with_namespace':project.path_with_namespace,
        'group_path':ns_d.get('full_path'),'web_url':project.web_url,
        'description':project.description,'visibility':project.visibility,
        'archived':bool(project.archived),
        'created_at':project.created_at,'last_activity_at':project.last_activity_at,
        'updated_at':project.updated_at,'default_branch':ref,
        'forks_count':int(project.forks_count) if project.forks_count is not None else None,
        'star_count':int(project.star_count) if project.star_count is not None else None,
        'open_issues_count':int(project.open_issues_count) if project.open_issues_count is not None else None,
        'team_name':extract_team_name(project.web_url),'package_json':'Yes',
    }
logger.info('fetch_project_with_pkg defined')


## 7 · Phase 2 worker — parse deps + scan imports

In [ ]:
def scan_project(row):
    """
    Returns (dep_rows, import_rows).
    dep_rows    : [{project_id, name, path, dependency}]  one row per package
    import_rows : [{project_id, name, path, import_filename, import_file_url, import_statement}]
                   one row per import statement in every source file
    """
    project_id=row['project_id']; name=row['name']; path=row['path']
    web_url=row['web_url']; ref=row.get('default_branch') or 'main'
    ns=row.get('path_with_namespace',path)
    project=row.get('_project')
    if project is None:
        with _gitlab_sem:
            try: project=client.projects.get(project_id)
            except Exception as e:
                logger.error('P2 project.get FAILED %s %s',ns,e); _inc('p2_errors'); return [],[]
    logger.info('P2 START | %s',ns)
    t0=time.perf_counter()
    base={'project_id':project_id,'name':name,'path':path}

    # --- dependencies ---
    dep_rows=[]
    pkg=_raw_file(project,'package.json',ref)
    if pkg:
        try:
            pj=json.loads(pkg)
            for tag in ('dependencies','devDependencies'):
                section=pj.get(tag,{})
                if not isinstance(section,dict): continue
                for dep_name in sorted(section.keys()):
                    dep_rows.append({**base,'tag':tag,'dependency':dep_name,'version':section[dep_name]})
            logger.info('P2 deps=%d | %s',len(dep_rows),ns)
        except Exception as e: logger.warning('P2 bad pkg.json %s %s',ns,e)
    _inc('dep_rows_total',len(dep_rows))

    # --- import statements ---
    all_items=_repo_tree_pages(project,ref)
    src_items=[i for i in all_items if i.get('type')=='blob']
    logger.info('P2 tree=%d blobs=%d | %s',len(all_items),len(src_items),ns)
    import_rows=[]
    lf=le=lerr=ls=0

    def process_file(item):
        nonlocal lf,le,lerr,ls
        fpath=item['path']
        content=_raw_file(project,fpath,ref)
        if content is None: lerr+=1; return None
        if not content.strip(): le+=1; return None
        lf+=1
        rows=[]
        for m in ANY_IMPORT_RE.finditer(content):
            rows.append({**base,
                'import_filename':fpath.split('/')[-1],
                'import_file_url':f'{web_url}/-/blob/{ref}/{fpath}',
                'import_statement':m.group(0).strip()})
            ls+=1
        return rows or None

    with ThreadPoolExecutor(max_workers=FILE_WORKERS) as pool:
        for r in pool.map(process_file,src_items):
            if r: import_rows.extend(r)

    logger.info('P2 DONE %.1fs files ok=%d empty=%d err=%d imports=%d | %s',
                time.perf_counter()-t0,lf,le,lerr,len(import_rows),ns)
    _inc('source_files_fetched',lf); _inc('source_files_empty',le)
    _inc('source_files_error',lerr); _inc('import_rows_total',len(import_rows))
    return dep_rows, import_rows

logger.info('scan_project defined')


## 8 · DB test

In [ ]:
t_total=time.perf_counter()
if not db_test_connection():
    raise RuntimeError('Cannot connect to Greenplum')


## 9 · Phase 1 gate

## 9b · Sample mode (runs instead of Phase 1/2 when SAMPLE_MODE=True)

In [ ]:
# =============================================================================
#  SAMPLE MODE — runs instead of the full Phase 1 / Phase 2 flow
# =============================================================================
if SAMPLE_MODE:
    print()
    print('=' * 70)
    print('  SAMPLE MODE — single project test')
    print(f'  Tables: {TBL_PROJECTS}')
    print(f'          {TBL_DEPS}')
    print(f'          {TBL_IMPORTS}')
    print('=' * 70)

    # Ask for the project_id
    while True:
        _pid_str = input('\n  Enter project_id to sample: ').strip()
        try: SAMPLE_PROJECT_ID = int(_pid_str); break
        except ValueError: print('  Please enter a numeric project_id.')

    logger.info('SAMPLE | fetching project %d ...', SAMPLE_PROJECT_ID)
    with _gitlab_sem:
        try: _sp = client.projects.get(SAMPLE_PROJECT_ID)
        except Exception as e:
            print(f'\n  ERROR: Could not fetch project {SAMPLE_PROJECT_ID}: {e}')
            raise SystemExit(1)

    logger.info('SAMPLE | found: %s', _sp.path_with_namespace)
    _ref = _sp.default_branch or 'main'
    _ns_d = _sp.namespace or {}
    _s_p1row = {
        '_project': _sp, '_ref': _ref,
        'project_id': int(_sp.id), 'name': _sp.name, 'path': _sp.path,
        'path_with_namespace': _sp.path_with_namespace,
        'group_path': _ns_d.get('full_path'), 'web_url': _sp.web_url,
        'description': _sp.description, 'visibility': _sp.visibility,
        'archived': bool(_sp.archived),
        'created_at': _sp.created_at, 'last_activity_at': _sp.last_activity_at,
        'updated_at': _sp.updated_at, 'default_branch': _ref,
        'forks_count': int(_sp.forks_count) if _sp.forks_count is not None else None,
        'star_count': int(_sp.star_count) if _sp.star_count is not None else None,
        'open_issues_count': int(_sp.open_issues_count) if _sp.open_issues_count is not None else None,
        'team_name': extract_team_name(_sp.web_url), 'package_json': 'Yes',
    }

    # Check package.json
    with _gitlab_sem:
        try: _sp.files.raw(file_path='package.json', ref=_ref); logger.info('SAMPLE | package.json found')
        except: logger.warning('SAMPLE | package.json NOT found on branch %s', _ref)

    print(f'\n  Project  : {_sp.path_with_namespace}')
    print(f'  Branch   : {_ref}')
    print(f'  web_url  : {_sp.web_url}')

    # Write project row
    print(f'\n  Writing project row to {TBL_PROJECTS} ...')
    conn = _get_conn()
    _create_table(conn, TBL_PROJECTS, PROJECTS_COLS, 'project_id')
    _bulk_insert(conn, TBL_PROJECTS, PROJECTS_COL_NAMES, [_s_p1row])
    conn.close(); print(f'  OK  1 row in {TBL_PROJECTS}')

    # Scan deps + imports
    print(f'\n  Scanning project for dependencies and imports ...')
    _s_dep_rows, _s_import_rows = scan_project(_s_p1row)

    # Write deps
    print(f'\n  Writing {len(_s_dep_rows)} dependency rows to {TBL_DEPS} ...')
    conn = _get_conn()
    _create_table(conn, TBL_DEPS, DEPS_COLS, 'project_id')
    _bulk_insert(conn, TBL_DEPS, DEPS_COL_NAMES, _s_dep_rows)
    conn.close(); print(f'  OK  {len(_s_dep_rows)} rows in {TBL_DEPS}')

    # Write imports
    print(f'\n  Writing {len(_s_import_rows)} import rows to {TBL_IMPORTS} ...')
    conn = _get_conn()
    _create_table(conn, TBL_IMPORTS, IMPORTS_COLS, 'project_id')
    _bulk_insert(conn, TBL_IMPORTS, IMPORTS_COL_NAMES, _s_import_rows)
    conn.close(); print(f'  OK  {len(_s_import_rows)} rows in {TBL_IMPORTS}')

    # Summary
    t_elapsed = time.perf_counter() - t_total
    print()
    print('=' * 70)
    print('  SAMPLE RUN COMPLETE')
    print('=' * 70)
    print(f'  Project ID  : {SAMPLE_PROJECT_ID}')
    print(f'  Path        : {_sp.path_with_namespace}')
    print(f'  {TBL_PROJECTS:<52}: 1 row')
    print(f'  {TBL_DEPS:<52}: {len(_s_dep_rows)} rows')
    print(f'  {TBL_IMPORTS:<52}: {len(_s_import_rows)} rows')
    print(f'  Elapsed (s) : {t_elapsed:.1f}')
    print('=' * 70)
    raise SystemExit('Sample run complete')


In [ ]:
print('\n'+'='*70)
print('  PHASE 1 — Identify UI projects (find package.json)')
print('='*70)
run_phase1=_ask('Run Phase 1? Scans GitLab and recreates gwma_ui_projects table')
p1_rows=[]
t_p1_elapsed=0.0


## 10 · Phase 1 execution (skipped if user said no)

In [ ]:
if run_phase1:
    logger.info('EXPORT fetching project list for group %s ...',GROUP_PATH)
    t_p1=time.perf_counter()
    group=client.groups.get(GROUP_PATH)
    proj_refs=group.projects.list(include_subgroups=True,all=True,per_page=PER_PAGE)
    total_all=len(proj_refs); run_stats['total_in_group']=total_all
    recent_refs=[p for p in proj_refs if is_recent(p)]
    old_count=total_all-len(recent_refs)
    run_stats['recent_projects']=len(recent_refs); run_stats['skipped_too_old']=old_count
    logger.info('DATE recent=%d too_old=%d cutoff=%s',len(recent_refs),old_count,UPDATED_SINCE.date())
    print(f'  Total: {total_all}  Recent: {len(recent_refs)}  Too old: {old_count}')
    print(f'\n  Checking package.json for {len(recent_refs)} projects ...\n')
    p1_done=0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        fmap={pool.submit(fetch_project_with_pkg,ref):ref for ref in recent_refs}
        for future in as_completed(fmap):
            p1_done+=1; ref=fmap[future]
            try: row=future.result(timeout=FUTURE_TIMEOUT)
            except FutureTimeout: _inc('p1_timeouts'); row=None
            except Exception as e: logger.error('P1 ERROR %s %s',ref.path_with_namespace,e); _inc('p1_errors'); row=None
            if row: p1_rows.append(row)
            if p1_done%PROGRESS_EVERY==0 or p1_done==len(recent_refs):
                pct=100*p1_done//len(recent_refs)
                print(f'\r  P1: {p1_done}/{len(recent_refs)} ({pct}%) '
                      f'| has_pkg={len(p1_rows)} no_pkg={run_stats["skipped_no_package_json"]} '
                      f'err={run_stats["p1_errors"]} timeout={run_stats["p1_timeouts"]}   ',
                      end='',flush=True)
    print()
    t_p1_elapsed=time.perf_counter()-t_p1
    logger.info('PHASE 1 done %.1fs rows=%d',t_p1_elapsed,len(p1_rows))
    # --- write to DB ---
    print(f'\n  Writing {len(p1_rows)} rows to {TBL_PROJECTS} ...')
    conn=_get_conn()
    _create_table(conn,TBL_PROJECTS,PROJECTS_COLS,'project_id')
    inserted=_bulk_insert(conn,TBL_PROJECTS,PROJECTS_COL_NAMES,p1_rows)
    conn.close()
    print(f'  OK  {inserted} rows in {TBL_PROJECTS}')
    print(f'  Projects with package.json : {len(p1_rows)}')
    print(f'  No package.json (skipped)  : {run_stats["skipped_no_package_json"]}')
    print(f'  Elapsed                    : {t_p1_elapsed:.1f}s')
else:
    print('  Skipping Phase 1 — using existing data in gwma_ui_projects.')


## 11 · Phase 2 gate

In [ ]:
print('\n'+'='*70)
print('  PHASE 2 — Parse dependencies & import statements')
print('='*70)
run_phase2=_ask('Run Phase 2? Parses package.json deps + scans source files for imports')
if not run_phase2:
    t_elapsed=time.perf_counter()-t_total
    print(f'Exiting. Total elapsed: {t_elapsed:.1f}s')
    raise SystemExit('Stopped at Phase 2 gate')


## 12 · Load project list for Phase 2

In [ ]:
if p1_rows:
    phase2_projects=p1_rows
    logger.info('P2 using %d in-memory Phase-1 rows',len(phase2_projects))
else:
    db_rows=db_load_projects()
    if not db_rows:
        raise RuntimeError(f'{TBL_PROJECTS} is empty — run Phase 1 first')
    phase2_projects=[{
        'project_id':int(r['project_id']),'name':r['name'],'path':r['path'],
        'path_with_namespace':r.get('path_with_namespace',r['path']),
        'web_url':r['web_url'],'default_branch':r['default_branch'] or 'main',
        '_project':None,
    } for r in db_rows]
    logger.info('P2 loaded %d projects from DB',len(phase2_projects))
print(f'  {len(phase2_projects)} projects to scan')


## 13 · Create Phase 2 tables

In [ ]:
conn=_get_conn()
_create_table(conn,TBL_DEPS,DEPS_COLS,'project_id')
_create_table(conn,TBL_IMPORTS,IMPORTS_COLS,'project_id')
conn.close()
print(f'Tables created: {TBL_DEPS}  |  {TBL_IMPORTS}')


## 14 · Phase 2 parallel scan

In [ ]:
t_p2=time.perf_counter()
all_dep_rows=[]; all_import_rows=[]; p2_done=0
print(f'  Scanning {len(phase2_projects)} projects ...\n')

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    fmap={pool.submit(scan_project,row):row for row in phase2_projects}
    for future in as_completed(fmap):
        p2_done+=1; row=fmap[future]
        ns=row.get('path_with_namespace',row.get('path','?'))
        try: dr,ir=future.result(timeout=FUTURE_TIMEOUT)
        except FutureTimeout: logger.warning('P2 TIMEOUT %s',ns); _inc('p2_timeouts'); dr,ir=[],[]
        except Exception as e: logger.error('P2 ERROR %s %s',ns,e); _inc('p2_errors'); dr,ir=[],[]
        all_dep_rows.extend(dr); all_import_rows.extend(ir)
        if p2_done%PROGRESS_EVERY==0 or p2_done==len(phase2_projects):
            pct=100*p2_done//len(phase2_projects)
            print(f'\r  P2: {p2_done}/{len(phase2_projects)} ({pct}%) '
                  f'| deps={len(all_dep_rows)} imports={len(all_import_rows)} '
                  f'err={run_stats["p2_errors"]} timeout={run_stats["p2_timeouts"]}   ',
                  end='',flush=True)
print()
t_p2_elapsed=time.perf_counter()-t_p2
logger.info('PHASE 2 done %.1fs deps=%d imports=%d',t_p2_elapsed,len(all_dep_rows),len(all_import_rows))


## 15 · Write Phase 2 to Greenplum

In [ ]:
print(f'  Writing {len(all_dep_rows)} dependency rows ...')
conn=_get_conn()
dep_inserted=_bulk_insert(conn,TBL_DEPS,DEPS_COL_NAMES,all_dep_rows)
conn.close()
print(f'  OK  {dep_inserted} rows in {TBL_DEPS}')

print(f'  Writing {len(all_import_rows)} import rows ...')
conn=_get_conn()
imp_inserted=_bulk_insert(conn,TBL_IMPORTS,IMPORTS_COL_NAMES,all_import_rows)
conn.close()
print(f'  OK  {imp_inserted} rows in {TBL_IMPORTS}')


## 16 · Final summary

In [ ]:
t_elapsed=time.perf_counter()-t_total
print()
print('='*70)
print('  FINAL SUMMARY')
print('='*70)
if run_phase1: print(f'  gwma_ui_projects rows     : {len(p1_rows)}')
print(f'  gwma_ui_dependencies rows : {len(all_dep_rows)}')
print(f'  gwma_ui_imports rows      : {len(all_import_rows)}')
print()
print(f'  Source files fetched      : {run_stats["source_files_fetched"]}')
print(f'  Phase 2 errors/timeouts   : {run_stats["p2_errors"]}/{run_stats["p2_timeouts"]}')
print(f'  Phase 1 elapsed (s)       : {t_p1_elapsed:.1f}')
print(f'  Phase 2 elapsed (s)       : {t_p2_elapsed:.1f}')
print(f'  Total elapsed (s)         : {t_elapsed:.1f}')
print('='*70)
print(f'  JOIN on project_id:')
print(f'    {TBL_PROJECTS}')
print(f'    {TBL_DEPS}')
print(f'    {TBL_IMPORTS}')
